0. What exactly we’ll measure

For each stock you’ve bought since 2017, we want:

    Invested amount (cost of current holdings)

    Current value

    Absolute return = (Current value − Invested amount)

    Return % = Absolute return / Invested amount

    Holding duration (years)

    Annualized return (CAGR) ≈ (Current value / Invested amount)^(1/years) − 1

Then you can:

    Sort by return % / CAGR → see which stocks accumulated the most value.

    Sort by negative or low return % → candidates to sell or review.

We’ll use:

    Tradebook CSVs → history, holding start dates, cashflows (if you want XIRR later).

    Holdings CSV from Zerodha → current quantity, average price, LTP (current price).

Tradebooks alone are not great for current value because you still need today’s prices. The holdings export already gives you LTP, saving a lot of pain.

In [2]:
# import libararies 

import pandas as pd 
from pathlib import Path

#1. Read and stack akk trandebook CSVs 

folder = Path("G:/My Drive/MyDatasets/MyInvests/tradebooks")
print (folder)
files = list (folder.glob("*.csv"))
print (files)

dfs = []
for f in files:
    df = pd.read_csv(f)
    df["source_file"] = f.name
    dfs.append(df)

trades_raw = pd.concat(dfs, ignore_index=True)


G:\My Drive\MyDatasets\MyInvests\tradebooks
[WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2019_20_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2020_21_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2021_22_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2022_23_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2023_24_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2024_25_tradebook-YG7149-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2017_18_tradebook-ZO7592-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2018_19_tradebook-ZO7592-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2019_20_tradebook-ZO7592-EQ.csv'), WindowsPath('G:/My Drive/MyDatasets/MyInvests/tradebooks/FY2020_21_tradebook-ZO7592-EQ.csv'), WindowsPath('G:

In [10]:
# print (trades_raw.head())
print (f"-----------===============---------------")
# print (trades_raw.describe)
print (f"-----------===============---------------")
# print (trades_raw.columns)
print (f"-----------===============---------------")
# print (trades_raw.value_counts)
print (f"-----------===============---------------")
print (trades_raw.dtypes)
print (f"-----------===============---------------")
print (f"-----------===============---------------")

-----------===============---------------
-----------===============---------------
-----------===============---------------
-----------===============---------------
symbol                   object
isin                     object
trade_date               object
exchange                 object
segment                  object
series                   object
trade_type               object
auction                    bool
quantity                float64
price                   float64
trade_id                  int64
order_id                  int64
order_execution_time     object
source_file              object
dtype: object
-----------===============---------------
-----------===============---------------


In [14]:
# 2. Normalize common column names (Zerodha naming varies a bit)
trades = trades_raw.copy()
print (trades.columns)
print (f"-----------===============---------------")
trades.columns = (
    trades.columns
    .str.strip()
    .str.lower()
    .str.replace(" ", "_")
)
print (trades.columns)
# rename_map = {
#     "trade_date": "date",
#     "exchange_time": "time",
#     "order_execution_time": "time",
#     "trading_symbol": "symbol",
#     "instrument": "symbol",
#     "transaction_type": "side",
#     "buy/sell": "side",
#     "quantity": "qty",
# }
# trades = trades.rename(columns={k: v for k, v in rename_map.items() if k in trades.columns})



Index(['symbol', 'isin', 'trade_date', 'exchange', 'segment', 'series',
       'trade_type', 'auction', 'quantity', 'price', 'trade_id', 'order_id',
       'order_execution_time', 'source_file'],
      dtype='object')
-----------===============---------------
Index(['symbol', 'isin', 'trade_date', 'exchange', 'segment', 'series',
       'trade_type', 'auction', 'quantity', 'price', 'trade_id', 'order_id',
       'order_execution_time', 'source_file'],
      dtype='object')


In [ ]:
# 3. Build datetime
if "time" in trades.columns:
    trades["datetime"] = pd.to_datetime(trades["date"] + " " + trades["time"])
else:
    trades["datetime"] = pd.to_datetime(trades["date"])

trades["side"] = trades["side"].str.upper().str.strip()
trades["qty"] = trades["qty"].astype(float)
trades["price"] = trades["price"].astype(float)

# If trade_value column exists, use it; else compute
if "trade_value" in trades.columns:
    trades["trade_value"] = trades["trade_value"].astype(float)
else:
    trades["trade_value"] = trades["price"] * trades["qty"]

In [11]:
# dfs[0]

df

,symbol,isin,trade_date,exchange,segment,series,trade_type,auction,quantity,price,trade_id,order_id,order_execution_time,source_file
0,CASTROLIND,INE172A01027,2024-07-02,BSE,EQ,A,sell,False,15.0,216.15,7843900,1719898690184621529,2024-07-02T11:52:18,FY2024_25_tradebook-VDZ078-EQ.csv
1,CASTROLIND,INE172A01027,2024-07-02,BSE,EQ,A,sell,False,85.0,216.10,7844100,1719898690184621529,2024-07-02T11:52:18,FY2024_25_tradebook-VDZ078-EQ.csv
2,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,122.0,169.80,2328719,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
3,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,52.0,169.80,2328720,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
4,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,235.0,169.80,2328721,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
5,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,141.0,169.80,2328722,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
6,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,200.0,169.80,2328723,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
7,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,89.0,169.78,2328724,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
8,CUB,INE491A01021,2024-09-04,NSE,EQ,EQ,sell,False,161.0,169.78,2328725,1000000013215184,2024-09-04T10:18:36,FY2024_25_tradebook-VDZ078-EQ.csv
